# 截断正态分布的最优拒绝抽样

> 对应代码：`MathStatsCode/code_in_notes/Chap.5/rejection_trunc_optimal.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：取最优指数参数 λ=(c+√(c²+4))/2 使接受率最大化，与 rejection_trunc.do 的固定 λ 对比。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.5`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.5")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页、设定种子并准备1000个候选观测（与 rejection_trunc.do 相同，唯一区别在工具分布参数的取法）。


In [ ]:
%%stata
clear
set more off
set seed 505
set obs 1000


**参数**

`scalar c=3`：设定截断点 c=3。


In [ ]:
%%stata
scalar c=3


**计算lambda和M**

计算归一化常数：`temp_cons=sqrt(2*pi)*(1-normal(c))`，即 √(2π)·(1-Φ(3))，用于计算包络常数 M。


In [ ]:
%%stata
scalar pi=3.1415926
scalar temp_cons=sqrt(2*pi)*(1-normal(c))


**最优lambda**

这里取**最优**的指数参数 λ：最大化接受率等价于最小化 M，对 M(λ) 关于 λ 求导可得驻点方程 λ²-cλ-1=0，即 λ=(c+√(c²+4))/2（本例 c=3 时约 3.303）。由于此时 λ>c，f/g 的最大值恰在 x=λ 处取得，因而 M 是最紧上界，**接受率最高**。可与前一个 notebook（λ=2.5）对比两者的接受率差异。


In [ ]:
%%stata
scalar lam=(c+sqrt(c^2+4))/2
scalar M=(exp(lam^2/2-lam*c-log(lam)))/temp_cons


**产生均匀分布和指数分布**

从工具分布抽样：x=c-ln(U)/λ（逆变换法）；u=runiform() 生成接受-拒绝比较的纵坐标。


In [ ]:
%%stata
gen x=c-log(runiform())/lam
gen u=runiform()


**截断正态分布密度函数**

计算目标密度：截断正态密度 f(x)=φ(x)/(1-Φ(c))，并加变量标签。


In [ ]:
%%stata
gen density=normalden(x)/(1-normal(c))
label variable density "f(x)"


**指数分布密度函数（用M标准化，即 M*g(x)）**

计算包络曲线 M·g(x)=M·λ·e^{-λ(x-c)}：因为 λ 最优，这条包络线应当紧贴着目标密度曲线的上缘（对比前一个 notebook 可以明显看出差别）。


In [ ]:
%%stata
gen density_exp=M*lam*exp(-1*lam*(x-c))
label variable density_exp "M*g(x)"


**挑出使得u<l/(M*g)的x，**

接受判据：`selected = u <= density/density_exp`，即 u ≤ f(x)/(M·g(x)) 时接受样本。


In [ ]:
%%stata
gen selected=u<=(density/density_exp)


**画图**

按 x 排序，便于折线图连线。


In [ ]:
%%stata
sort x


**显示方便，散点图y轴用标准化的密度函数值乘以u**

画图并导出：四层图——包络线（虚线）、目标密度（黑线）、拒绝点（叉号）、接受点（对号）。把两张图（固定 λ 与最优 λ）放在一起对比，可以直观理解“工具分布的参数如何影响拒绝率”：最优 λ 下包络线更紧，落在曲线外（被拒绝）的点明显更少。


In [ ]:
%%stata
gen u_y=u*density_exp
twoway (line density_exp x, lpattern(dash)) ///
     (line density x, lc(black)) ///
     (scatter u_y x if selected==0, msize(small) msymbol(x) mc(gs8)) ///
     (scatter u_y x if selected==1, msize(small) msymbol(v) mc(gs10)) ///
     , legend(on ring(0) pos(1)  ///
     label(3 "Rejected") label(4 "Accepted") ///
     label(2 "Density of Accepted")) ///
     graphr(fcolor(white) color(white))
